In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset = pd.read_csv('insurance_pre.csv')

In [3]:
dataset

,age,sex,bmi,children,smoker,charges
0,19,female,27.900,0,yes,16884.92400
1,18,male,33.770,1,no,1725.55230
2,28,male,33.000,3,no,4449.46200
3,33,male,22.705,0,no,21984.47061
4,32,male,28.880,0,no,3866.85520
...,...,...,...,...,...,...
1333,50,male,30.970,3,no,10600.54830
1334,18,female,31.920,0,no,2205.98080
1335,18,female,36.850,0,no,1629.83350
1336,21,female,25.800,0,no,2007.94500


In [4]:
datasets=pd.get_dummies(dataset,dtype=int,drop_first=True)

In [5]:
datasets

,age,bmi,children,charges,sex_male,smoker_yes
0,19,27.900,0,16884.92400,0,1
1,18,33.770,1,1725.55230,1,0
2,28,33.000,3,4449.46200,1,0
3,33,22.705,0,21984.47061,1,0
4,32,28.880,0,3866.85520,1,0
...,...,...,...,...,...,...
1333,50,30.970,3,10600.54830,1,0
1334,18,31.920,0,2205.98080,0,0
1335,18,36.850,0,1629.83350,0,0
1336,21,25.800,0,2007.94500,0,0


In [8]:
indep=datasets[['age', 'sex_male', 'bmi', 'children', 'smoker_yes']]
dep=datasets[['charges']]

In [9]:
#split into training set and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(indep, dep, test_size = 1/3, random_state = 0)

In [10]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [11]:
from sklearn.ensemble import RandomForestRegressor

In [17]:
from sklearn.model_selection import GridSearchCV
param_grid = {
        'criterion':['squared_error','absolute_error'],
        'max_features': ['sqrt','log2'],
        'n_estimators':[10,100]
}
grid = GridSearchCV(RandomForestRegressor(), param_grid, refit = True, verbose = 3,n_jobs=-1)
grid.fit(X_train, y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits


C:\Users\swadh\anaconda4\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


,estimator,RandomForestRegressor()
,param_grid,"{'criterion': ['squared_error', 'absolute_error'], 'max_features': ['sqrt', 'log2'], 'n_estimators': [10, 100]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,100


In [18]:
# print best parameter after tuning
#print(grid.best_params_)
re=grid.cv_results_
#print(re)
grid_predictions = grid.predict(X_test)
# print classification report
from sklearn.metrics import r2_score
r_score=r2_score(y_test,grid_predictions)
print("The R_score value for best parameter {}:".format(grid.best_params_),r_score)

The R_score value for best parameter {'criterion': 'squared_error', 'max_features': 'sqrt', 'n_estimators': 100}: 0.875377182671226


In [19]:
table=pd.DataFrame.from_dict(re)

In [20]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_criterion,param_max_features,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.380914,0.044277,0.053978,0.004371,squared_error,sqrt,10,"{'criterion': 'squared_error', 'max_features':...",0.801556,0.765130,0.827373,0.794426,0.746849,0.787067,0.028240,6
1,1.685867,0.039664,0.102008,0.003269,squared_error,sqrt,100,"{'criterion': 'squared_error', 'max_features':...",0.810005,0.768863,0.839114,0.828742,0.759549,0.801255,0.031793,1
2,0.225469,0.101295,0.028803,0.010528,squared_error,log2,10,"{'criterion': 'squared_error', 'max_features':...",0.790093,0.750844,0.828281,0.812806,0.754385,0.787282,0.030823,5
3,1.376417,0.008416,0.068992,0.004056,squared_error,log2,100,"{'criterion': 'squared_error', 'max_features':...",0.802847,0.771123,0.830463,0.829500,0.760770,0.798941,0.028892,4
4,0.377197,0.035949,0.024846,0.004610,absolute_error,sqrt,10,"{'criterion': 'absolute_error', 'max_features'...",0.770060,0.741266,0.823284,0.813370,0.750415,0.779679,0.033048,8
5,2.456671,0.057962,0.044038,0.005244,absolute_error,sqrt,100,"{'criterion': 'absolute_error', 'max_features'...",0.802693,0.759351,0.840901,0.831690,0.762875,0.799502,0.033805,2
6,0.307190,0.007124,0.015574,0.001171,absolute_error,log2,10,"{'criterion': 'absolute_error', 'max_features'...",0.792458,0.757189,0.815127,0.800105,0.764213,0.785818,0.021880,7
7,2.273407,0.050795,0.036050,0.002274,absolute_error,log2,100,"{'criterion': 'absolute_error', 'max_features'...",0.795629,0.778828,0.836050,0.821379,0.765070,0.799391,0.026238,3


In [21]:
age_input=float(input("Age:"))
bmi_input=float(input("BMI:"))
children_input=float(input("Children:"))
sex_male_input=int(input("Sex Male 0 or 1:"))
smoker_yes_input=int(input("Smoker Yes 0 or 1:"))

Age: 20
BMI: 25
Children: 5
Sex Male 0 or 1: 2
Smoker Yes 0 or 1: 5


In [22]:
Future_Prediction=grid.predict([[age_input,bmi_input,children_input,sex_male_input,smoker_yes_input]])
print("Future_Prediction={}".format(Future_Prediction))

Future_Prediction=[48244.0902245]
